# Obtain FAFB SynapticCoordinates
The aim here is to go from a single root number to get a list of the synaptic coordinates (pre-synaptic inputs only).

Note 
1. You will need to use the `fau_connectomics` environment stored in the `environment.yml`
2. You will also need to have a token to access the FAFB programatically
    * You can use the guide to get a token if you don't have one.
    * I've stored the token in the .env file (you can create your own .env following the example)

In [ ]:
#libraries
import os
import pandas as pd
from decouple import config, Config, RepositoryEnv
from caveclient import CAVEclient
from getpass import getpass

# Get variables from the .env file
ENV_PATH = "../.env"
config = Config(RepositoryEnv(ENV_PATH))
# Get the CAVE_AUTH_TOKEN
cave_token = config("CAVE_AUTH_TOKEN", default=None)
if not cave_token:
    print("No CAVE token found in the .env file.")
    temp_token = getpass("Token not found in the .env file. Please enter your CAVE token (if you don't have one leave if blank and instructions will appear): ")
    if temp_token:
        cave_token = temp_token
    else:
        print("No CAVE token entered. Follow the information below to get a token.")
        CAVEclient.auth.get_new_token()

if cave_token:
    # Initialize the CAVE client
    client = CAVEclient('flywire_fafb_public')
    auth = client.auth
    auth.save_token(cave_token)

In [ ]:
client.info.get_datastacks()

In [ ]:
root_id= "720575940624928574"
pre_df = client.materialize.query_table(
      table='synapses_nt_v1',
      filter_in_dict={'pre_pt_root_id': [root_id]},
  )
display(pre_df.head())

In [ ]:
root_id= "720575940624928574"

def get_presynaptic_targets(client, root_id, vx_resolution=None):
  pre_df = client.materialize.query_table(
      table='synapses_nt_v1',
      filter_in_dict={'pre_pt_root_id': [root_id]},
  )
  #Get voxel resolution if not pre-defined
  if vx_resolution is None:
      vx_resolution = client.materialize.get_table_metadata('synapses_nt_v1')["voxel_resolution"]

  #Initial info
  output_df = pre_df[["pre_pt_root_id", "post_pt_root_id", "pre_pt_position", "pre_pt_supervoxel_id"]].copy()
  output_df = output_df.rename(columns={
      "pre_pt_root_id": "root_id",
      "post_pt_root_id": "post_synaptic_root_id",
      "pre_pt_position": "xyz_voxel",
      "pre_pt_supervoxel_id": "supervoxel_id"
  })

  output_df["voxel_resolution_nm"] = [vx_resolution] * output_df.shape[0]

  return output_df

voxel_res = client.materialize.get_table_metadata("synapses_nt_v1")["voxel_resolution"]
root_presynapses = get_presynaptic_targets(client, root_id, voxel_res)
root_presynapses.to_parquet("../data/test_root_presynapses.parquet", index=False)
display(root_presynapses)

In [ ]:
# Debug: Check current working directory and .env file
import os
from decouple import config, Config, RepositoryEnv

print(f"Current working directory: {os.getcwd()}")
print(f".env file exists in current directory: {os.path.exists('.env')}")
print(f".env file exists in parent directory: {os.path.exists('../.env')}")

# Specify the path to the .env file in the parent directory
env_path = os.path.join(os.path.dirname(os.getcwd()), '.env')
config = Config(RepositoryEnv(env_path))

# Read the CAVE_AUTH_TOKEN
cave_token = config("CAVE_AUTH_TOKEN", default=None)
print(f"Your CAVE token is: {cave_token}")

# Set the token as environment variable for CAVEclient
if cave_token:
    os.environ["CAVE_AUTH_TOKEN"] = cave_token
    print("CAVE token successfully loaded and set!")
else:
    print("Warning: CAVE token not found in .env file")